In [ ]:
import os
import chromadb
# 💡 改從 ollama 引入 Client 類別，能更穩定地控制連線
from ollama import Client 
from PIL import Image

# 💡 強制讓本機請求繞過任何可能開啟的 VPN 或網路代理 (Proxy)
os.environ['NO_PROXY'] = '127.0.0.1,localhost'

# 💡 初始化 Ollama 客戶端，明確指定本地伺服器位址與埠口
ollama_client = Client(host='http://127.0.0.1:11434')

def get_vlm_description(image_path, model_name):
    """利用 VLM 分析圖片內容"""
    prompt = "請詳細描述這張花卉照片的特徵，包括花種、顏色、外觀細節以及適合的送禮場合。請用繁體中文回答。"
    
    # 💡 使用剛剛建立的 ollama_client 發送請求
    response = ollama_client.generate(model=model_name, prompt=prompt, images=[image_path])
    return response['response']

def build_index_with_vlm(IMAGE_FOLDER, db_path, collection_name, model_name):
    # 初始化 ChromaDB 本地持久化客戶端
    client = chromadb.PersistentClient(path=db_path)
    collection = client.get_or_create_collection(name=collection_name)
    
    # 讀取資料夾內所有圖片
    image_files = [f for f in os.listdir(IMAGE_FOLDER) if f.endswith(('.jpg', '.jpeg', '.png'))]
    
    if not image_files:
        print(f"⚠️ 警告：在 {IMAGE_FOLDER} 資料夾中找不到任何圖片檔案。")
        return

    for filename in image_files:
        path = os.path.join(IMAGE_FOLDER, filename)        
        
        # 1. 提取檔名基本資訊（例如 "向日葵_1.jpg" 切割出 "向日葵"）
        flower_name = filename.split('_')[0]        
        
        # 2. 透過 VLM 獲取深度特徵 (這就是檢索的依據)
        print(f"正在分析 {filename}...")
        try:
            vlm_description = get_vlm_description(path, model_name)        
            
            # 3. 存入 ChromaDB
            # 將「自動生成的描述」與「檔名」結合作為可以用來語意搜尋的關鍵文字
            combined_text = f"品種:{flower_name}。特徵分析:{vlm_description}"        
            
            collection.add(
                documents=[combined_text],
                 metadatas=[{"flower_type": flower_name, "filename": filename}],
                ids=[filename]
            )
        except Exception as e:
            print(f"❌ 分析 {filename} 時發生錯誤: {e}")
            
    print(f"🎉 全部圖片分析並索引完成！")

if __name__ == "__main__":
    # 執行前請確保這兩個資料夾路徑正確，且 img_db 內有你的花卉圖片
    build_index_with_vlm(
        IMAGE_FOLDER="./img_db", 
        db_path="./chroma_flower_db", 
        collection_name="flowers", 
        model_name="qwen2.5vl-cpu:latest"
    )


正在分析 向日葵_1.jpg...
